# DortDB Benchmarks: TPC-H

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

LOG_PATH = '../../dist/logs/'
GRAPH_PATH = '../../dist/graphs/'
os.makedirs(GRAPH_PATH, exist_ok=True)

dort = pd.concat([
    pd.read_json(f'{LOG_PATH}{path}.log', lines=True) for path in [
        'tpch_dortdb_1-2-3-4-5-6-7-8-9-10-11-12-13-14-15-16-17-18-20-21-22'
    ]
], ignore_index=True)
dort_old = pd.read_json(f'{LOG_PATH}tpch_dortdb_old.log', lines=True)
sqlite = pd.read_json(f'{LOG_PATH}tpch_sqlite_1-2-3-4-5-6-7-8-9-10-11-12-13-14-15-16-17-18-19-20-21-22.log', lines=True)
alasql_old = pd.read_json(f'{LOG_PATH}alasql.log', lines=True)
alasql = pd.concat([
    pd.read_json(f'{LOG_PATH}{path}.log', lines=True) for path in [
        'tpch_alasql_1-2-3-4-5-6',
        'tpch_alasql_7-8-9-10-11-12-13',
        'tpch_alasql_11-13',
        'tpch_alasql_14-15-16-17-18-19',
        'tpch_alasql_20-21-22',
    ]
], ignore_index=True)

for df in [dort, dort_old, sqlite, alasql, alasql_old]:
    df['query_temp'] = ''
    for row in df.itertuples():
        if hasattr(row, 'query') and not pd.isna(row.query):
            df.at[row.Index, 'query_temp'] = f'q{int(row.query):02d}'
        elif isinstance(row.detail, dict) and 'q' in row.detail:
            df.at[row.Index, 'query_temp'] = f'q{row.detail['q']:02d}'
    df['query'] = df['query_temp']
    df.drop(columns=['query_temp'], inplace=True)

# colors = {
#     'DortDB': '#005cbb',
#     'SQL.js': '#42b983',
#     'AlaSQL': '#f7df1e',
# }
colors = {
    'DortDB': '#e5ccff',
    'DortDB (old)': "#cea0ff",
    'SQL.js': "#9ec79a",
    # 'AlaSQL': '#ea6b66',
    'AlaSQL': "#f8ef69",
    'AlaSQL (old)': "#e95b56",
}
labels = {
    'dort': 'DortDB',
    'dort_old': 'DortDB (old)',
    'sqlite': 'SQL.js',
    'alasql': 'AlaSQL',
    'alasql_old': 'AlaSQL (old)',
}

## Execution time

In [ ]:
for (df, label) in zip([dort, alasql], ['DortDB', 'AlaSQL']):
    for q in df['query'].dropna().unique():
        mask_q = df['query'] == q
        # detect any non-warmup runs (either via column or detail dict)
        warmed_detail = df.loc[mask_q, 'detail'].apply(lambda d: isinstance(d, dict) and d.get('isWarmup') == False)
        if (warmed_detail).any():
            # print(f'Dropping warmup runs for query {q} in dataframe {label}')
            # drop warmup runs for this query (column or detail)
            warmup_detail = df['detail'].apply(lambda d: isinstance(d, dict) and d.get('isWarmup') == True)
            warmup_mask = mask_q & warmup_detail
            df.drop(df[warmup_mask].index, inplace=True)

dort_extime = dort[dort['msg'] == 'Performance entry'].groupby('query').agg({'duration': 'mean'}) / 1000
# dort_old_extime = dort_old[dort_old['msg'] == 'Performance entry'].groupby('query').agg({'duration': 'mean'}) / 1000
sqlite_extime = sqlite[sqlite['msg'] == 'Performance entry'].groupby('query').agg({'duration': 'mean'}) / 1000
alasql_extime = alasql[alasql['msg'] == 'Performance entry'].groupby('query').agg({'duration': 'mean'}) / 1000
# alasql_old_extime = alasql_old[alasql_old['msg'] == 'Performance entry'].groupby('query').agg({'duration': 'mean'}) / 1000

extime = pd.concat([
    dort_extime.rename(columns={'duration': 'DortDB'}),
    # dort_old_extime.rename(columns={'duration': 'DortDB (old)'}),
    sqlite_extime.rename(columns={'duration': 'SQL.js'}),
    alasql_extime.rename(columns={'duration': 'AlaSQL'}),
    # alasql_old_extime.rename(columns={'duration': 'AlaSQL (old)'}),
], axis=1)
extime.sort_index(inplace=True)

color_list = [colors[col] for col in extime.columns]
ax = extime.plot(kind='bar', width=0.8, color=color_list)

for bar in ax.patches[:len(extime)]:
        bar.set_edgecolor('black')
        bar.set_zorder(3)

alasql_error_queries = ['q02', 'q07', 'q08', 'q15', 'q22']
alasql_old_error_queries = ['q07', 'q20', 'q22']

incorrect_queries = [
    df[df['msg'] == 'Query result does NOT match expected result']['query'].unique() for df in [dort, sqlite, alasql]
]

plt.gcf().set_size_inches(12, 4)
for i, row in enumerate(extime.itertuples()):
    for j, val in enumerate(row[1:]):
        if pd.isna(val):
            if j == 2 and row.Index in alasql_error_queries:
                ax.text(i - 0.375 + j * (0.8 / len(extime.columns)), 0.002, 'ERROR',
                    ha='left', va='baseline', rotation=90, color='red', fontsize=10)
            else:
                if j == 0:
                    ax.bar(i - 0.395 + j * (0.8 / len(extime.columns)), 3600, 
                    width=0.9 / len(extime.columns), 
                    color='white', edgecolor='black', align='edge')
                ax.bar(i - 0.375 + j * (0.8 / len(extime.columns)), 3600, 
                width=0.75 / len(extime.columns), 
                color='white', edgecolor=color_list[j], hatch='//', align='edge')
        elif row.Index in incorrect_queries[j]:
            ax.text(i - 0.375 + j * (0.8 / len(extime.columns)), 0.002, 'INCORRECT',
                    ha='left', va='bottom', rotation=90, color='black', fontsize=10)

ax.set_ylabel('Execution Time (seconds)')
ax.set_xlabel('Query')
ax.set_yscale('log')
# ax.legend(loc='upper right')
ax.legend(bbox_to_anchor=(0., 1.02, 1., .102), loc='lower left',
                      ncols=3, mode="expand", borderaxespad=0.)
ax.set_ylim(None, 3600)
ax.tick_params('x', rotation=0)

plt.tight_layout()
plt.savefig(f'{GRAPH_PATH}tpch-execution.pdf', bbox_inches='tight')
plt.show()